# 1. Generate sample queries and descriptions (SQL to Text)

> 이 노트북은 [dongjin-ml/text2sql Git repo](https://github.com/dongjin-ml/text2sql/blob/main/lab3_text2sql_schema_preparation/1.sample_queries.ipynb)를 참고하여 작성되었습니다. 

복잡한 데이터베이스에서 Text2SQL의 가장 어려운 작업은 쿼리 생성에 필요한 스키마를 선별하는 과정, 즉 Schema Linking 입니다.

현실의 기업 환경에서는 테이블/컬럼 이름이 의미를 축약하고 있어서 LLM이 이를 파악하기 힘들거나, 테이블/컬럼이 너무 많아서 모든 목록을 프롬프트에 담아 전달하는 것이 불가능한 경우가 많습니다.

이를 해결하기 위해, 우리 DB에 맞춰 스키마 설명 문서를 정제하고, LLM에 필요한 컨텍스트를 선별하여 제공하는 작업이 필요합니다. 

이 노트북에서는, CUR 데이터 분석에 자주 사용되는 [샘플 쿼리 23개](./cur_sample_queries.sql)를 기준으로 각 쿼리가 어떤 상황에 사용되어야 하는지에 대한 description을 작성하여 LLM이 추후 text2sql 작업을 할 때 참고할 수 있는 데이터로 가공합니다. 또한 이를 키워드 및 vector 형태로 Opensearch에 인덱싱하여, 추후 text2sql 챗봇이 관련 쿼리를 검색하여 참고할 수 있도록 합니다. 

전체 작업 흐름은 아래와 같이 이어갈 예정입니다. (이 노트북에서는 아래 그림의 1 / 3 과정을 수행합니다. 2는 불필요하여 생략합니다)


![Intro](./img/schema-prep.png)


## Step 0: OpenSearch 환경 설정

In [62]:
!pip install -U opensearch-py langchain_aws langchain_community --quiet

In [2]:
from libs.ssm import parameter_store
pm = parameter_store('us-east-1')
domain_endpoint = pm.get_params(key="opensearch_domain_endpoint", enc=False)
opensearch_domain_endpoint = f"https://{domain_endpoint}"
opensearch_user_id = pm.get_params(key="opensearch_user_id", enc=False)
opensearch_user_password = pm.get_params(key="opensearch_user_password", enc=True)
print(opensearch_domain_endpoint)

https://https://search-finops-rag-test-2kfmtuqcqx4wd5xhepyn2atu3q.us-east-1.es.amazonaws.com


## Step 1: Schema Description 문서 로드 (위 그림의 `1. Schema Loader`)

각 기업에는 Excel / CSV 등으로 스키마 설명 문서가 정의되어 있을 수 있습니다. 이를 Parsing하여 아래의 Schema Description 포맷으로 변경한다고 가정하겠습니다.

```
{
    "table_name": {
        "table_desc": "Description of the table",
        "cols": [
            {
                "col": "Column Name 1",
                "col_desc": "Description of the column including PK info"
            },
            {
                "col": "Column Name 2",
                "col_desc": "Description of the column"
            }
        ]
    }
}
```

초기 설명 문서에는 테이블의 이름과 테이블에 대한 기본 설명, 컬럼 이름과 컬럼에 대한 설명이 포함되어야 합니다. 기업에 잘 정리된 스키마 설명 문서가 없다면, 아주 기본적인 정보만 제공하고 LLM이 이를 증강하여 초기 설명문서 자체를 생성하도록 할 수도 있습니다. 이를 위한 LLM 호출 스크립트는 다음 [링크](https://github.com/kevmyung/db-schema-loader/blob/main/schema_loader.py)를 참고합니다.

In [6]:
import json

file_path = './cur_schema.json'

with open(file_path, 'r', encoding='utf-8') as file:
    schema_description = json.load(file)

print(json.dumps(schema_description, indent=4, ensure_ascii=False))

[
    {
        "cur.hourly_view_all": {
            "table_desc": "AWS CUR 데이터를 시간별로 집계한 뷰",
            "cols": [
                {
                    "col": "product_code",
                    "col_desc": "측정된 제품의 코드입니다. 예를 들어 Amazon EC2는 Amazon Elastic Compute Cloud의 제품 코드입니다."
                },
                {
                    "col": "service",
                    "col_desc": "고객에 대한 특정 AWS 서비스를 고유한 짧은 약어로 식별합니다. (예시: Amazon EC2, AWS KMS)"
                },
                {
                    "col": "operation",
                    "col_desc": "이 항목에서 다루는 특정 AWS 작업입니다. 이 항목의 구체적인 사용량에 대해 설명합니다. 예를 들어 RunInstances 값은 Amazon EC2 인스턴스 작업을 나타냅니다."
                },
                {
                    "col": "charge_type",
                    "col_desc": "이 항목에 적용되는 요금 유형입니다. 가능한 유형은 다음과 같습니다. (참고: charge_type이 할인인 항목의 경우 BlendedCost는 비어 있습니다. 할인은 멤버 계정의 일반 비용만 사용하여 계산되며 멤버 계정 및 SKU별로 집계됩니다. 따라서 BlendedCost는 할인에 사용할 수 없습니다)"
                },
                {
           

In [12]:
import json
import os
from langchain_aws import ChatBedrock
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

_SYS_PROMPT_TEMPLATE_2 = """
당신은 AWS 비용 분석 전문가입니다. 제공된 컬럼 정보를 기반으로 더 자세한 설명과 활용 방안을 제시해주세요.

작업 지침:
- 컬럼의 세부적인 의미와 용도를 설명하세요
- 비용 분석과 최적화에서 이 컬럼을 어떻게 활용할 수 있는지 설명하세요
- AWS 비용 분석 관점에서 이 컬럼의 중요성을 설명하세요
- 컬럼명은 변경하지 마세요
- 아래 JSON 형식을 정확히 따르세요

출력 형식:
{
    "col": "컬럼명",
    "col_desc": "기본 설명",
    "detailed_info": {
        "purpose": "상세 용도 및 의미",
        "cost_analysis_usage": ["비용 분석 활용 방안 1", "활용 방안 2"]
    }
}
"""

_USER_PROMPT_TEMPLATE = """
<컬럼_정보>
컬럼명: {col}
현재설명: {col_desc}
</컬럼_정보>
"""

model_kwargs =  { 
    "max_tokens": 200000,
    "temperature": 0.0,
    "top_k": 250,
    "top_p": 1
}

# CUR 스키마 파일 로드
with open('cur_schema.json', 'r', encoding='utf-8') as file:
    cur_schema = json.load(file)

if not os.path.exists('cur_metadata'):
    os.makedirs('cur_metadata')

usr_prompt = ChatPromptTemplate.from_template(_USER_PROMPT_TEMPLATE)

model_kwargs["system"] = _SYS_PROMPT_TEMPLATE_2
model = ChatBedrock(model_id="anthropic.claude-3-sonnet-20240229-v1:0", region_name='us-east-1', model_kwargs=model_kwargs)
chain = usr_prompt | model | StrOutputParser()

# 컬럼들에 대해 처리
enhanced_columns = []

for col_info in cur_schema[0]['cur.hourly_view_all']['cols']:
    response = chain.invoke({
        "col": col_info['col'],
        "col_desc": col_info['col_desc']
    })
    enhanced_columns.append(json.loads(response))

# 결과를 원본 구조와 동일하게 구성
enhanced_schema = [{
    'cur.hourly_view_all': {
        'table_desc': cur_schema[0]['cur.hourly_view_all']['table_desc'],
        'cols': enhanced_columns
    }
}]

# 결과 저장
with open('./cur_metadata/enhanced_cur_schema.json', 'w', encoding='utf-8') as file:
    json.dump(enhanced_schema, file, ensure_ascii=False, indent=2)


### 이제 Schema Description 문서를 활용해 후속 작업을 이어가겠습니다

## Step 2: SQL2Text 샘플 쿼리 변환 (위 그림의 `2. Query Translator`)

Lab 1 / Lab 2에서 언급했듯이, 좋은 샘플 쿼리를 LLM에게 제공하는 것은 쿼리 작성 뿐만 아니라 Schema Linking에도 도움이 됩니다.

그러나, 대부분의 기업 환경에서 자주 사용되는 쿼리를 로그로 관리하고 있는 반면, (기존에 Text2SQL을 사용하지 않았기 때문에) 쿼리에 매칭되는 자연어 질문은 없습니다. 

Step 2에서는 자주 사용하는 쿼리들을 자연어 질문으로 변환하는 SQL2Text 과정을 진행합니다.

In [15]:
sql_file = './cur_sample_queries.sql'

with open(sql_file, 'r') as file:
    data = file.read()

queries = [query.strip() for query in data.split(';') if query.strip()]

for i, query in enumerate(queries, start=1):
    print(f"Query {i}:\n{query}\n{'-'*80}\n")

Query 1:
SELECT linked_account_id, SUM(unblended_cost) FROM cur.hourly_view_all GROUP BY 1 ORDER BY 2 DESC
--------------------------------------------------------------------------------

Query 2:
SELECT DATE_TRUNC('month', usage_date) as month, SUM(unblended_cost) FROM cur.hourly_view_all GROUP BY 1
--------------------------------------------------------------------------------

Query 3:
SELECT product_code, COUNT(*) FROM cur.hourly_view_all WHERE unblended_cost > 100 GROUP BY product_code
--------------------------------------------------------------------------------

Query 4:
SELECT product_code, SUM(unblended_cost) FROM cur.hourly_view_all GROUP BY product_code
--------------------------------------------------------------------------------

Query 5:
SELECT DATE_TRUNC('day', usage_date) as usage_day, SUBSTRING(usage_type, 1, 4) as region, SUM(usage_quantity) as instance_hours FROM cur.hourly_view_all WHERE usage_type LIKE '%BoxUsage%' GROUP BY DATE_TRUNC('day', usage_date), SUBS

쿼리를 해석하기 위해, 각 쿼리에 사용된 테이블/컬럼의 의미를 파악해야 합니다.
따라서, 각 쿼리에 사용된 테이블/컬럼 정보를 아래와 같이 추출합니다.
```
{
  "table": ["table1", "table2", ...],
  "column": ["col1", "col2", ...]
}
```
다음은 SQL 쿼리에 활용된 스키마 목록을 추출하는 LLM 요청 구문입니다.

In [16]:
SYS_PROMPT_TEMPLATE1 = """ 
You are an expert in extracting table names and column names from SQL queries. 
From the provided SQL query, extract all table names and column names used for SELECT, WHERE, and JOIN clauses, excluding asterisks ("*"). 
Ensure that the response is in a valid JSON format that can be used directly with json.load(). 
Skip the preamble and only provide the answer in a JSON document:

{
  "table": ["table1", "table2", ...],
  "column": ["col1", "col2", ...]
}

<example>
SQL:
SELECT * from LOGIS_ADMIN.IAWD_TB_DCBSCD_BASISLC_M 
where basis_lclsf_cd_nm like '%예약구분%'
LIMIT 200;

{
  "table": ["IAWD_TB_DCBSCD_BASISLC_M"],
  "column": ["basis_lclsf_cd_nm"]
}
</example>
"""

USR_PROMPT_TEMPLATE1="""
SQL: {sql}
"""

In [17]:
from langchain_aws import ChatBedrock
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [18]:
model_kwargs =  { 
    "max_tokens": 200000,
    "temperature": 0.0,
    "top_k": 250,
    "top_p": 1
}

In [19]:
model_kwargs["system"] = SYS_PROMPT_TEMPLATE1
model1 = ChatBedrock(model_id="us.anthropic.claude-3-5-sonnet-20241022-v2:0", region_name='us-west-2', model_kwargs=model_kwargs)
prompt1 = ChatPromptTemplate.from_template(USR_PROMPT_TEMPLATE1)

chain1 = prompt1 | model1 | StrOutputParser()

예를 들어 아래 쿼리에 사용된 스키마를 추출해보겠습니다.

```SELECT usage_date, operation, SUM(usage_quantity) as usage_amount FROM cur.hourly_view_all WHERE operation IN ('RunInstances', 'CreateVolume', 'CreateSnapshot') GROUP BY usage_date, operation ORDER BY usage_date DESC``` 

In [20]:
sql = queries[8].strip()
response = chain1.invoke({"sql": sql})
used_schema = json.loads(response)
print(used_schema)

{'table': ['hourly_view_all'], 'column': ['usage_date', 'operation', 'usage_quantity']}


#### 이제 이 쿼리에 사용된 스키마 설명을 조회합니다.

In [48]:
def extract_descriptions(table_info, tables, columns):
    tables_lower = {table.lower() for table in tables}
    columns_lower = {column.lower() for column in columns}
    
    description = {
        "table": {},
        "column": {}
    }
    
    for cur_schema in table_info:
        for table_name, table_info in cur_schema.items():
            description["table"][table_name] = table_info["table_desc"]
            for col in table_info["cols"]:
                col_name = col["col"]
                if col_name.lower() in columns_lower:
                    description["column"][col_name] = col["col_desc"]
    return description

In [49]:
extracted_description = extract_descriptions(schema_description, used_schema['table'], used_schema['column'])
print(extracted_description)

{'table': {'cur.hourly_view_all': 'AWS CUR 데이터를 시간별로 집계한 뷰'}, 'column': {'operation': '이 항목에서 다루는 특정 AWS 작업입니다. 이 항목의 구체적인 사용량에 대해 설명합니다. 예를 들어 RunInstances 값은 Amazon EC2 인스턴스 작업을 나타냅니다.', 'usage_date': '이 항목의 시작 날짜와 시간(UTC)입니다. 포괄적입니다. 형식은 YYYY-MM-DDTHH:mm:ssZ입니다.', 'usage_quantity': '지정된 기간 동안 발생한 사용량의 총합입니다.'}}


#### 이제 쿼리에 대한 자연어 변환을 요청합니다.

In [50]:
SYS_PROMPT_TEMPLATE2 = """ 
당신은 AWS CUR(Cost and Usage Report) 데이터 분석 전문가입니다.
주어진 SQL 쿼리의 의도와 비즈니스 목적을 파악하여 자연스러운 한국어로 설명해주세요.

작업 지침:
- CUR 데이터의 특성을 고려한 실무적인 관점에서 설명
- AWS 비용 최적화 및 분석 관점에서 해당 쿼리의 목적 설명
- 쿼리가 도출하는 데이터의 실제 활용 방안 제시
- 비용 분석/최적화 담당자가 이해하기 쉬운 자연스러운 한국어로 표현
- 모든 조건절과 집계 방식의 비즈니스 의미를 포함
- 불필요한 서론 없이 핵심적인 내용만 간단명료하게 서술

출력 형식:
1. 쿼리 의도: [실무자가 이 데이터를 요청하는 실제 상황]
2. 비즈니스 목적: [이 데이터가 비용 최적화에 기여하는 방식]
3. 실무 활용: [데이터 기반 의사결정 및 조치 사항]
4. 연관 분석: [후속 분석이나 타 지표와의 연계 방안]
"""

USR_PROMPT_TEMPLATE2="""
<쿼리_설명>
{description}
</쿼리_설명>

CUR 쿼리문: {sql}
"""

In [51]:
model_kwargs["system"] = SYS_PROMPT_TEMPLATE2
model2 = ChatBedrock(model_id="us.anthropic.claude-3-5-sonnet-20241022-v2:0", region_name='us-west-2', model_kwargs=model_kwargs)
prompt2 = ChatPromptTemplate.from_template(USR_PROMPT_TEMPLATE2)
chain2 = prompt2 | model2 | StrOutputParser()

#### 자연어 질문을 생성하는 프롬프트는 아래 형식으로 LLM에 전달됩니다.

In [52]:
print(SYS_PROMPT_TEMPLATE2)
print(prompt2.format(description=extracted_description, sql=queries[8]))

 
당신은 AWS CUR(Cost and Usage Report) 데이터 분석 전문가입니다.
주어진 SQL 쿼리의 의도와 비즈니스 목적을 파악하여 자연스러운 한국어로 설명해주세요.

작업 지침:
- CUR 데이터의 특성을 고려한 실무적인 관점에서 설명
- AWS 비용 최적화 및 분석 관점에서 해당 쿼리의 목적 설명
- 쿼리가 도출하는 데이터의 실제 활용 방안 제시
- 비용 분석/최적화 담당자가 이해하기 쉬운 자연스러운 한국어로 표현
- 모든 조건절과 집계 방식의 비즈니스 의미를 포함
- 불필요한 서론 없이 핵심적인 내용만 간단명료하게 서술

출력 형식:
1. 쿼리 의도: [실무자가 이 데이터를 요청하는 실제 상황]
2. 비즈니스 목적: [이 데이터가 비용 최적화에 기여하는 방식]
3. 실무 활용: [데이터 기반 의사결정 및 조치 사항]
4. 연관 분석: [후속 분석이나 타 지표와의 연계 방안]

Human: 
<쿼리_설명>
{'table': {'cur.hourly_view_all': 'AWS CUR 데이터를 시간별로 집계한 뷰'}, 'column': {'operation': '이 항목에서 다루는 특정 AWS 작업입니다. 이 항목의 구체적인 사용량에 대해 설명합니다. 예를 들어 RunInstances 값은 Amazon EC2 인스턴스 작업을 나타냅니다.', 'usage_date': '이 항목의 시작 날짜와 시간(UTC)입니다. 포괄적입니다. 형식은 YYYY-MM-DDTHH:mm:ssZ입니다.', 'usage_quantity': '지정된 기간 동안 발생한 사용량의 총합입니다.'}}
</쿼리_설명>

CUR 쿼리문: SELECT usage_date, operation, SUM(usage_quantity) as usage_amount FROM cur.hourly_view_all WHERE operation IN ('RunInstances', 'CreateVolume', 'CreateSnapshot') GROUP BY usage_date, operation ORDER 

In [53]:
response = chain2.invoke({"sql": queries[8], "description": extracted_description})
print(response)

1. 쿼리 의도:
EC2 관련 핵심 리소스(인스턴스, EBS 볼륨, 스냅샷) 생성 작업의 시간별 사용량을 추적하는 쿼리입니다. 특히 운영팀이 인프라 확장 패턴을 파악하고자 할 때 주로 사용됩니다.

2. 비즈니스 목적:
- EC2 인스턴스, EBS 볼륨, 스냅샷의 생성 트렌드를 시계열로 분석
- 리소스 생성 패턴을 통한 비정상적인 리소스 증가 감지
- 인프라 확장이 계획된 범위 내에서 이루어지는지 모니터링

3. 실무 활용:
- 예상치 못한 리소스 생성 급증 시 즉각적인 조사 진행
- 부서별/프로젝트별 리소스 할당량 준수 여부 확인
- 자동 스케일링 설정의 적절성 검증
- 스냅샷 생성 정책의 효율성 평가

4. 연관 분석:
- 리소스 생성량과 실제 비용 발생 간의 상관관계 분석
- 리소스 생성 패턴과 애플리케이션 사용량 비교
- 리소스 유형별 생성 대비 삭제 비율 모니터링
- 특정 시간대/요일별 리소스 생성 패턴 분석을 통한 비용 최적화 기회 발굴


#### 다음 쿼리에 대한 자연어 설명은 LLM에 의해 위와 같이 정의되었습니다.

```SELECT usage_date, operation, SUM(usage_quantity) as usage_amount FROM cur.hourly_view_all WHERE operation IN ('RunInstances', 'CreateVolume', 'CreateSnapshot') GROUP BY usage_date, operation ORDER BY usage_date DESC```

#### 아래는 위 과정을 모든 SQL 쿼리에 대해 반복하는 스크립트입니다. (약 1~2분 소요됩니다)

In [54]:
import os

FILE_PATH_1 = './cur_sample_queries_desc.jsonl'
def query_translation(table_info, queries, chain1, chain2):
    if os.path.exists(FILE_PATH_1):
        os.remove(FILE_PATH_1)

    with open(FILE_PATH_1, 'a') as output_file:
        for query in queries:
            sql = query.strip()
            
            try:
                response = chain1.invoke({"sql": sql})
                schema = json.loads(response)
            except json.JSONDecodeError:
                print(response)
                time.sleep(1)  

            description = extract_descriptions(table_info, schema["table"], schema["column"])
            
            input = chain2.invoke({"sql": sql, "description": description})
            # Write input and query to the file in JSON format
            data = {"input": input, "query": sql}
            output_file.write(json.dumps(data, ensure_ascii=False) + "\n")
            
query_translation(schema_description, queries, chain1, chain2)

#### 쿼리 변환이 완료된 결과는 `./cur_sample_queries_desc.jsonl` 파일에 저장되어 있습니다. 

In [55]:
with open(FILE_PATH_1, 'r') as file:
    for line in file:
        data = json.loads(line)
        print(data)

{'input': '1. 쿼리 의도:\nAWS 조직 내 모든 계정들의 총 비용을 계정별로 집계하여, 비용 발생이 가장 큰 계정부터 순서대로 파악하고자 합니다. 할인이나 크레딧이 적용되기 전의 실제 사용량 기준 비용(unblended_cost)을 사용하여 순수한 리소스 사용 비용을 확인합니다.\n\n2. 비즈니스 목적:\n- 조직 내 비용 책임 소재를 명확히 파악\n- 비용 발생이 큰 계정들을 식별하여 우선적인 비용 최적화 대상 선정\n- 계정별 비용 할당 및 내부 비용 정산의 기초 데이터로 활용\n- 부서/프로젝트별 예산 관리 및 비용 통제를 위한 기준 수립\n\n3. 실무 활용:\n- 비용이 높은 상위 계정들에 대한 상세 분석 진행\n- 계정별 비용 한도 설정 및 예산 알림 구성\n- 부서별/프로젝트별 비용 배분 및 차지백(Chargeback) 정책 수립\n- 비정상적으로 높은 비용이 발생하는 계정 모니터링\n\n4. 연관 분석:\n- 계정별 주요 사용 서비스 분석으로 확장\n- 시계열 분석을 통한 계정별 비용 추세 파악\n- 태그 기반 분석과 연계하여 프로젝트/환경별 세부 분석\n- RI/Savings Plan 활용률과 연계하여 계정별 비용 최적화 기회 발굴', 'query': 'SELECT linked_account_id, SUM(unblended_cost) FROM cur.hourly_view_all GROUP BY 1 ORDER BY 2 DESC'}
{'input': '1. 쿼리 의도:\n월별 AWS 총 비용을 파악하기 위한 기본적인 비용 집계 쿼리입니다. 할인이나 크레딧이 적용되기 전의 순수 사용량 기준 비용(unblended_cost)을 월 단위로 합산하여 실제 리소스 사용에 따른 원가를 파악합니다.\n\n2. 비즈니스 목적:\n- 월간 AWS 비용 추이를 모니터링하여 비정상적인 비용 증가 감지\n- 예산 계획 수립 및 실제 지출 현황 비교 분석\n- 전월 대비 비용 변동성 파악을 통한 비용 예측 기초 자료 확보\n\n3. 실무 활용:\n- 월별 

## Step 3: 샘플 쿼리 벡터 임베딩 및 OpenSearch 저장

이제 <자연어 질문 & SQL 쿼리> 조합의 자연어 질문을 벡터로 임베딩하여, 사용자 질문과 유사한 SQL 쿼리를 찾아내기 용이하도록 저장해야 합니다.

아래 구문은 OpenSearch 환경을 초기화합니다. (연결 생성 및 Index 초기화)

In [59]:
import yaml
from opensearchpy import OpenSearch, RequestsHttpConnection
INDEX_NAME = "example_queries"

def load_opensearch_config():
    with open("./libs/opensearch.yml", 'r', encoding='utf-8') as file:
        return yaml.safe_load(file)

def init_opensearch(config):
    mapping = {"settings": config['settings'], "mappings": config['mappings-sql']}
    endpoint = opensearch_domain_endpoint
    http_auth = (opensearch_user_id, opensearch_user_password)

    os_client = OpenSearch(
            hosts=[{'host': endpoint.replace("https://", ""),'port': 443}],
            http_auth=http_auth, 
            use_ssl=True,
            verify_certs=True,
            timeout=300,
            connection_class=RequestsHttpConnection
    )

    create_os_index(os_client, mapping)
    return os_client

def create_os_index(os_client, mapping):
    exists = os_client.indices.exists(INDEX_NAME)

    if exists:
        os_client.indices.delete(index=INDEX_NAME)
        print("Existing index has been deleted. Create new one.")
    else:
        print("Index does not exist, Create one.")

    os_client.indices.create(INDEX_NAME, body=mapping)

config = load_opensearch_config()
os_client = init_opensearch(config)

Index does not exist, Create one.


RequestError: RequestError(400, 'illegal_argument_exception', 'Unknown tokenizer type [nori_tokenizer] for [nori_tokenizer]')

이제 앞에 만들었던 <자연어 질문 & SQL 쿼리>를 벡터 임베딩으로 변환하고, OpenSearch에 bulk indexing 할 수 있는 Data-Action 포맷으로 구성합니다.

In [19]:
from langchain_community.embeddings import BedrockEmbeddings

FILE_PATH_2 = './cur_example_queries.jsonl'
emb_model = BedrockEmbeddings(model_id="amazon.titan-embed-text-v2:0", region_name='us-west-2', model_kwargs={"dimensions":1024}) 

def input_embedding(emb_model):
    num = 0
    if os.path.exists(FILE_PATH_2):
        os.remove(FILE_PATH_2)

    with open(FILE_PATH_1, 'r') as input_file, open(FILE_PATH_2, 'a') as output_file:
        for line in input_file:
            data = json.loads(line)
            input = data['input']
            query = data['query']
            
            # Data part
            body = { "input": input, "query": query, "input_v": emb_model.embed_query(input) }

            # Action part
            action = { "index": { "_index": INDEX_NAME, "_id": str(num) } }

            # Write action and body to the file in correct bulk format
            output_file.write(json.dumps(action, ensure_ascii=False) + "\n")
            output_file.write(json.dumps(body, ensure_ascii=False) + "\n")

            num += 1    

input_embedding(emb_model)

/tmp/ipykernel_5954/1904691214.py:4: LangChainDeprecationWarning: The class `BedrockEmbeddings` was deprecated in LangChain 0.2.11 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-aws package and should be used instead. To use it run `pip install -U :class:`~langchain-aws` and import as `from :class:`~langchain_aws import BedrockEmbeddings``.
  emb_model = BedrockEmbeddings(model_id="amazon.titan-embed-text-v2:0", region_name='us-west-2', model_kwargs={"dimensions":1024})


#### 위 코드를 실행한 뒤 `./cur_example_queries.jsonl` 파일을 열어보면, 변환된 임베딩을 확인할 수 있습니다.

In [20]:
with open(FILE_PATH_2, 'r') as file:
    bulk_data = file.read()
        
response = os_client.bulk(body=bulk_data)
if response["errors"]:
    print("There were errors during bulk indexing:")
    for item in response["items"]:
        if 'index' in item and item['index']['status'] >= 400:
            print(f"Error: {item['index']['error']['reason']}")
else:
    print("Bulk-inserted all items successfully.")

Bulk-inserted all items successfully.


#### 이제 OpenSearch에 저장을 완료했습니다.